# Radiomics-Based Thyroid Nodule Classification with Label-Conditional Conformal Prediction

This notebook implements the pipeline described in the paper submitted to CIBB 2026.

**Dataset**: [TN5000](https://figshare.com/s/cb6a67f17c04b29e7edd) - upload `TN5000.zip` to your Google Drive and set the paths in the *Path Configuration* cell below.

**Reproducibility**: All intermediate results (features, scaled data, metrics) are saved to `RESULTS_DIR` on Google Drive at the end of each section, enabling the pipeline to be resumed from any checkpoint. To run the full pipeline from scratch, execute all cells sequentially.

**Structure**:
- *Dataset Import and Preprocessing* - optional if you use the precomputed features
- *PyRadiomics Feature Extraction* - optional, precomputed features available in `FEATURES_DIR`
- *Feature Selection* - Spearman filter + z-score standardization
- *Classification* - LR-LASSO, GAM, XGBoost, Random Forest, ExtraTrees, MLP
- *GAM Interpretation* - partial dependence plots, permutation importance
- *Label-Conditional Cross-Conformal Prediction (CCCP)*

# Dataset Import and Preprocessing

Step 0: Drive connection and PyRadiomics install

In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Installing PyRadiomics from source
!git clone https://github.com/Radiomics/pyradiomics.git
!pip install SimpleITK -q
!cd pyradiomics && pip install . -q

# Installing PyGAM
!pip install pygam -q

Step 1: Module Imports

In [ ]:
# Standard library
import os
import zipfile
import xml.etree.ElementTree as ET
import pickle
import logging
import re

# Data handling
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# Image processing
import cv2
import SimpleITK as sitk
from tqdm import tqdm

# Scikit-learn
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

# ML models
from xgboost import XGBClassifier
from pygam import LogisticGAM

# PyRadiomics
import radiomics
from radiomics import featureextractor

# Check PyRadiomics Version
print(radiomics.__version__)

# Suppress PyRadiomics verbose logs -> show errors only
logging.getLogger('radiomics').setLevel(logging.ERROR)

Step 2: Path Configuration

In [ ]:
# Google Drive paths -> adjust to your setup
ZIP_PATH      = "/content/drive/MyDrive/TN5000.zip"
EXTRACT_PATH  = "/content/TN5000"
DATASET_PATH  = "/content/TN5000/TN5000_forReview"
FEATURES_DIR  = "/content/drive/MyDrive/TN5000_features"
RESULTS_DIR   = "/content/drive/MyDrive/thyroid_radiomics_results"

# Derived paths -> do not modify
IMAGES_DIR      = os.path.join(DATASET_PATH, "JPEGImages")
ANNOTATIONS_DIR = os.path.join(DATASET_PATH, "Annotations")
SETS_DIR        = os.path.join(DATASET_PATH, "ImageSets", "Main")

# Create output directories
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

Step 3: Data extraction from ZIP and import on Colab

In [ ]:
# Data Extraction
os.makedirs(EXTRACT_PATH, exist_ok=True)

print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print("Unzip completed")

# Verify directory structure
for d in [IMAGES_DIR, ANNOTATIONS_DIR, SETS_DIR]:
    print(f"{d}: {len(os.listdir(d))} files")

Step 4: Label and Bounding Box Extraction from XML Annotations

In [ ]:
def parse_annotation(xml_path):
    """
    Read a XML PASCAL VOC file in TN5000.
    Returns label (0=benign, 1=malignant) and bbox coordinates.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    obj   = root.find('object')
    label = int(obj.find('name').text)  # 0=benign, 1=malignant

    bbox  = obj.find('bndbox')
    xmin  = int(float(bbox.find('xmin').text))
    ymin  = int(float(bbox.find('ymin').text))
    xmax  = int(float(bbox.find('xmax').text))
    ymax  = int(float(bbox.find('ymax').text))

    return label, xmin, ymin, xmax, ymax

# Test on a single file
sample_xml = os.path.join(ANNOTATIONS_DIR,
             os.listdir(ANNOTATIONS_DIR)[0])
label, xmin, ymin, xmax, ymax = parse_annotation(sample_xml)
print(f"\nTest parsing XML: label={label}, "
      f"bbox=({xmin},{ymin},{xmax},{ymax})")


Step 5: Loading dataset splits

In [ ]:
def load_split(split_name):
    txt_path = os.path.join(SETS_DIR, f"{split_name}.txt")
    with open(txt_path, 'r') as f:
        ids = [line.strip() for line in f if line.strip()]
    return ids

train_ids = load_split('train')
val_ids   = load_split('val')
test_ids  = load_split('test')

print(f"\nSplits:")
print(f"  Train: {len(train_ids)}")
print(f"  Val:   {len(val_ids)}")
print(f"  Test:  {len(test_ids)}")

Step 6: Dataset Construction

In [ ]:
def build_record(img_id):
    '''
    Returns a dictionary for each record containing the image id and path,
    the label, the coordinates for the bounding box and height andt width of the image
    '''
    img_path = os.path.join(IMAGES_DIR, f"{img_id}.jpg")
    xml_path = os.path.join(ANNOTATIONS_DIR, f"{img_id}.xml")

    img = cv2.imread(img_path)
    if img is None:
        print(f" Image not found: {img_path}")
        return None

    h, w = img.shape[:2]
    label, xmin, ymin, xmax, ymax = parse_annotation(xml_path)

    return {
        'id'       : img_id,
        'img_path' : img_path,
        'label'    : label,
        'xmin'     : xmin,
        'ymin'     : ymin,
        'xmax'     : xmax,
        'ymax'     : ymax,
        'img_w'    : w,
        'img_h'    : h,
    }

# Construction of training, validation and test sets
datasets = {}

for split_name, ids in [('train', train_ids),
                         ('val', val_ids),
                         ('test', test_ids)]:
    records = []
    for i in ids:
        r = build_record(i)
        if r is not None:
            records.append(r)
    datasets[split_name] = records
    labels = [r['label'] for r in records]
    print(f"  {split_name}: {len(records)} images "
          f"| Malignant={sum(labels)} "
          f"Benign={len(labels)-sum(labels)}")

# DataFrame Conversion
df_train = pd.DataFrame(datasets['train'])
df_val   = pd.DataFrame(datasets['val'])
df_test  = pd.DataFrame(datasets['test'])

print(f"\n DataFrame Columns: {list(df_train.columns)}")

Step 7: Visualization example

In [ ]:
# SINGLE SAMPLE

def plot_sample(row, save=False):
    """
    Prints entire image with highlighted bounding box.
    """
    img = cv2.imread(row['img_path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    label_str = 'Malignant' if row['label'] == 1 else 'Benign'
    color     = 'red'       if row['label'] == 1 else 'blue'

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    ax.imshow(img, cmap='gray')
    rect = patches.Rectangle(
        (row['xmin'], row['ymin']),
        row['xmax'] - row['xmin'],
        row['ymax'] - row['ymin'],
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    ax.set_title(f"{label_str} — ID: {row['id']}",
                 fontsize=12, color=color)
    ax.axis('off')

    plt.tight_layout()
    if save:
      plt.savefig(os.path.join(RESULTS_DIR, f"sample_{row['id']}.png"),
                dpi=150, bbox_inches='tight')
    plt.show()

# Examples
benign  = df_train[df_train['label'] == 0]
malignant  = df_train[df_train['label'] == 1]
plot_sample(benign.iloc[4])
plot_sample(malignant.iloc[25])

In [ ]:
# SAMPLE PAIR

def plot_sample_pair(benign_row, malignant_row, save=False):
    """
    Plots a benign and malignant sample side by side.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, row, label_str, color in zip(
        axes,
        [benign_row, malignant_row],
        ['Benign', 'Malignant'],
        ['blue', 'red']
    ):
        img = cv2.imread(row['img_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ax.imshow(img, cmap='gray')
        rect = patches.Rectangle(
            (row['xmin'], row['ymin']),
            row['xmax'] - row['xmin'],
            row['ymax'] - row['ymin'],
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.set_title(label_str, fontsize=13, color=color)
        ax.axis('off')

    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(RESULTS_DIR, "sample_pair.png"),
                dpi=150, bbox_inches='tight')
    plt.show()

# Example
benign    = df_train[df_train['label'] == 0]
malignant = df_train[df_train['label'] == 1]
plot_sample_pair(benign.iloc[5], malignant.iloc[25], save = True)

# PyRadiomics Feature Extraction



Check [PyRadiomics](
https://pyradiomics.readthedocs.io/en/latest/features.html) website to better understand

Step 0: Extractor Configuration

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()

# Only Original
extractor.disableAllImageTypes()
extractor.enableImageTypeByName('Original')

#  Explicitly enable features
extractor.disableAllFeatures()
extractor.enableFeatureClassByName('firstorder')
extractor.enableFeatureClassByName('shape2D')
extractor.enableFeatureClassByName('glcm')
extractor.enableFeatureClassByName('glrlm')
extractor.enableFeatureClassByName('glszm')
extractor.enableFeatureClassByName('gldm')
extractor.enableFeatureClassByName('ngtdm')

# Settings
extractor.settings['force2D']         = True
extractor.settings['binWidth'] = 25           # PyRadiomics default
extractor.settings['normalize']       = True
extractor.settings['normalizeScale']  = 100
extractor.settings['preCrop']         = True

# Disables resampling (original resolution)
extractor.settings['resampledPixelSpacing'] = None

print("Configured extractor")

# Verify enabled features
print(f"Enabled features: {list(extractor.enabledFeatures.keys())}")

Step 1: Feature Extraction Functions

In [ ]:
# ----------------- Extraction function -------------------

def extract_features(row):
    img = cv2.imread(row['img_path'], cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    h, w = img.shape
    mask = np.zeros((h, w), dtype=np.uint8)
    mask[row['ymin']:row['ymax'],
         row['xmin']:row['xmax']] = 1

    sitk_img  = sitk.GetImageFromArray(img.astype(np.float32))
    sitk_mask = sitk.GetImageFromArray(mask)

    try:
        result = extractor.execute(sitk_img, sitk_mask)
        features = {
            k: float(v)
            for k, v in result.items()
            if not k.startswith('diagnostics_')
        }
        return features
    except Exception as e:
        print(f"Error on {row['id']}: {e}")
        return None


# ----------------- Extraction on entire split -------------------

def extract_all(df, split_name):
    all_features = []
    failed = []

    for _, row in tqdm(df.iterrows(),
                       total=len(df),
                       desc=f"Extraction {split_name}"):
        feats = extract_features(row)
        if feats is not None:
            feats['id']    = row['id']
            feats['label'] = row['label']
            all_features.append(feats)
        else:
            failed.append(row['id'])

    df_out = pd.DataFrame(all_features)
    print(f"\n{split_name}: {len(df_out)} extracted, "
          f"{len(failed)} failed, "
          f"{len(df_out.columns)-2} features")
    return df_out


Step 2: Feature Extraction and Save

In [ ]:
# -------------- EXTRACTION ------------------

df_feat_train = extract_all(df_train, 'train')
df_feat_val   = extract_all(df_val, 'val')
df_feat_test  = extract_all(df_test, 'test')


# Extracted features
feature_cols = [c for c in df_feat_train.columns
                if c not in ['id', 'label']]
print(f"Total Features: {len(feature_cols)}")
print(f"Feature Example: {feature_cols[:5]}")


# -------------- DRIVE SAVE ------------------

os.makedirs(FEATURES_DIR, exist_ok=True)

df_feat_train.to_csv(os.path.join(FEATURES_DIR, 'features_train.csv'), index=False)
df_feat_val.to_csv(  os.path.join(FEATURES_DIR, 'features_val.csv'),   index=False)
df_feat_test.to_csv( os.path.join(FEATURES_DIR, 'features_test.csv'),  index=False)

print(f"\nSaved in: {FEATURES_DIR}")

# Feature Selection

Step 0: Load Features
\
  *Run only if you skipped PyRadiomics extraction above*

In [ ]:
df_train = pd.read_csv(os.path.join(FEATURES_DIR, 'features_train.csv'))
df_val   = pd.read_csv(os.path.join(FEATURES_DIR, 'features_val.csv'))
df_test  = pd.read_csv(os.path.join(FEATURES_DIR, 'features_test.csv'))


Step 1: Separate features from labels

In [ ]:
feature_cols = [c for c in df_train.columns if c not in ['id', 'label']]

X_train = df_train[feature_cols].values
y_train = df_train['label'].values
X_val   = df_val[feature_cols].values
y_val   = df_val['label'].values
X_test  = df_test[feature_cols].values
y_test  = df_test['label'].values

print(f"Number of features: {len(feature_cols)}")
print(f"Training:   {X_train.shape} | Malignant: {y_train.sum()} Benign: {(y_train==0).sum()}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

Step 2: Feature class inspection

In [ ]:
family_counts = {}
for col in feature_cols:

    match = re.match(r'original_(\w+?)_', col)
    if match:
        family = match.group(1)
        family_counts[family] = family_counts.get(family, 0) + 1

for family, count in sorted(family_counts.items()):
    print(f"{family}: {count}")
print(f"\nTotal: {sum(family_counts.values())}")

Step 3: Feature Selection by Spearman correlation filter

In [ ]:
def spearman_filter(X, feature_names, threshold=0.95):
    """
    Removes features with Spearman correlation > threshold.
    For each correlated pair, keeps the first and removes the second.
    """
    n_features = X.shape[1]

    # compute correlation matrix
    corr_matrix = np.zeros((n_features, n_features))
    for i in range(n_features):
        for j in range(i+1, n_features):
            r, _ = spearmanr(X[:, i], X[:, j])
            corr_matrix[i, j] = abs(r)
            corr_matrix[j, i] = abs(r)

    # features to remove
    to_remove = set()
    for i in range(n_features):
        if i in to_remove:
            continue
        for j in range(i+1, n_features):
            if j in to_remove:
                continue
            if corr_matrix[i, j] > threshold:
                to_remove.add(j)

    to_keep = [i for i in range(n_features) if i not in to_remove]
    return to_keep, corr_matrix

# ---  Spearman filter on raw data ---
th = 0.90 # set threshold
print(f"Spearman filter (threshold={th})")

feature_names = np.array(feature_cols)  # from your original loading cell
kept_idx, corr_matrix = spearman_filter(X_train, feature_names, threshold=th)

X_train_spe = X_train[:, kept_idx]
X_val_spe   = X_val[:, kept_idx]
X_test_spe  = X_test[:, kept_idx]
kept_spe    = feature_names[kept_idx]

print(f"  Features before: {X_train.shape[1]}")
print(f"  Features after:  {X_train_spe.shape[1]}")
print(f"  Removed:         {X_train.shape[1] - X_train_spe.shape[1]}")
print(f"\nKept features:")
for f in kept_spe:
    print(f"  {f}")

# correlation heatmap
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap='coolwarm', vmin=0, vmax=1)
plt.colorbar(label='|Spearman r|')
plt.title('Spearman correlation matrix (pre-filtering)')
plt.tight_layout()
plt.show()

Step 4: Feature Standardization (z-score feature by feature)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_spe)
X_val_sc   = scaler.transform(X_val_spe)
X_test_sc = scaler.transform(X_test_spe)

print(f"\nStandardization completed")


Check number of features retained for each family

In [ ]:
family_counts_selected = {}
for col in kept_spe:
    match = re.match(r'original_(\w+?)_', col)
    if match:
        family = match.group(1)
        family_counts_selected[family] = family_counts_selected.get(family, 0) + 1

print("Feature families after Spearman filtering:")
for family, count in sorted(family_counts_selected.items()):
    print(f"  {family}: {count}")
print(f"\nTotal: {sum(family_counts_selected.values())}")

Step 5: Save Features and Labels to Drive

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

np.save(os.path.join(RESULTS_DIR, 'kept_spe.npy'),    kept_spe)
np.save(os.path.join(RESULTS_DIR, 'X_train_sc.npy'),  X_train_sc)
np.save(os.path.join(RESULTS_DIR, 'X_val_sc.npy'),    X_val_sc)
np.save(os.path.join(RESULTS_DIR, 'X_test_sc.npy'),   X_test_sc)
np.save(os.path.join(RESULTS_DIR, 'y_train.npy'),     y_train)
np.save(os.path.join(RESULTS_DIR, 'y_val.npy'),       y_val)
np.save(os.path.join(RESULTS_DIR, 'y_test.npy'),      y_test)

print("=" * 55)
print(" FILES SAVED ON DRIVE")
print("=" * 55)
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size  = os.path.getsize(fpath)
    unit  = 'KB' if size < 1e6 else 'MB'
    size  = size/1e3 if size < 1e6 else size/1e6
    print(f"  {fname:<35} {size:6.1f} {unit}")
print("=" * 55)

# Classification

Step 0:  Data Loading and Metrics functions

In [ ]:
# Run only if you skipped the Feature Selection section above
kept_spe   = np.load(os.path.join(RESULTS_DIR, 'kept_spe.npy'))
X_train_sc = np.load(os.path.join(RESULTS_DIR, 'X_train_sc.npy'))
X_val_sc   = np.load(os.path.join(RESULTS_DIR, 'X_val_sc.npy'))
X_test_sc  = np.load(os.path.join(RESULTS_DIR, 'X_test_sc.npy'))
y_train    = np.load(os.path.join(RESULTS_DIR, 'y_train.npy'))
y_val      = np.load(os.path.join(RESULTS_DIR, 'y_val.npy'))
y_test     = np.load(os.path.join(RESULTS_DIR, 'y_test.npy'))

print("Loading finished.")

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    """Returns dictionary with AUC, Sens, Spec, Prec, NPV, F1."""
    y_pred = (y_prob >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1   = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
    return dict(AUC=auc, Sens=sens, Spec=spec, Prec=prec, NPV=npv, F1=f1,
                TP=tp, TN=tn, FP=fp, FN=fn)

def print_metrics(name, split, m):
    print(f"\n{'─'*55}")
    print(f"  {name}  —  {split}")
    print(f"{'─'*55}")
    print(f"  AUC:  {m['AUC']:.4f}   Sens: {m['Sens']:.4f}   Spec: {m['Spec']:.4f}")
    print(f"  Prec: {m['Prec']:.4f}   NPV:  {m['NPV']:.4f}   F1:   {m['F1']:.4f}")
    print(f"  Confusion matrix:")
    print(f"           Pred Benign   Pred Malignant")
    print(f"  True Benign   {m['TN']:5d}    {m['FP']:5d}")
    print(f"  True Malignant   {m['FN']:5d}    {m['TP']:5d}")


Classifier 1:  Logistic Regression + LASSO penalization

In [ ]:
print("Fitting LR-LASSO...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lasso_selector = LogisticRegressionCV(
    Cs=np.logspace(-4, 2, 50),
    cv=cv,
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
lasso_selector.fit(X_train_sc, y_train)

# Selected features
coef         = lasso_selector.coef_[0]
selected_idx = np.where(coef != 0)[0]
kept_lasso   = feature_names[selected_idx]


print(f"  Features: {X_train_sc.shape[1]} -> {len(kept_lasso)}")

# Metrics on validation set
lr_val  = compute_metrics(y_val,  lasso_selector.predict_proba(X_val_sc)[:,1])

print_metrics("LR-LASSO", "Validation", lr_val)


print(f"\n{kept_lasso}")

Classifier 2: GAM Regression

In [ ]:
print("\nFitting GAM...")
sw_train = compute_sample_weight('balanced', y_train)   # class balance

gam = LogisticGAM(n_splines=10)
gam.fit(X_train_sc, y_train, weights=sw_train)

gam_val  = compute_metrics(y_val,  gam.predict_proba(X_val_sc))
print_metrics("GAM", "Validation", gam_val)


Classifier 3: XGBoost

In [ ]:
print("\nFitting XGBoost...")
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()   # class balance

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train_sc, y_train)

xgb_val  = compute_metrics(y_val,  xgb.predict_proba(X_val_sc)[:,1])
print_metrics("XGBoost", "Validation", xgb_val)


Classifier 4: Random Forest

In [ ]:
print("\nFitting Random Forest...")
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)

rf_val  = compute_metrics(y_val,  rf.predict_proba(X_val_sc)[:,1])
print_metrics("RandomForest", "Validation", rf_val)

Classifier 5: ExtraTrees

In [ ]:
print("\nFitting ExtraTrees...")
et = ExtraTreesClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
et.fit(X_train_sc, y_train)

et_val  = compute_metrics(y_val,  et.predict_proba(X_val_sc)[:,1])
print_metrics("ExtraTrees", "Validation", et_val)

Classifier 6: MLP

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

mlp.fit(X_train_sc, y_train)

mlp_val = compute_metrics(y_val, mlp.predict_proba(X_val_sc)[:,1])
print_metrics("MLP", "Validation", mlp_val)

Summary (on validation set)

In [ ]:
proba_fn = {
    'LR-LASSO'    : lambda X: lasso_selector.predict_proba(X)[:,1],
    'GAM'         : lambda X: gam.predict_proba(X),
    'XGBoost'     : lambda X: xgb.predict_proba(X)[:,1],
    'RandomForest': lambda X: rf.predict_proba(X)[:,1],
    'ExtraTrees'  : lambda X: et.predict_proba(X)[:,1],
    'MLP'         : lambda X: mlp.predict_proba(X)[:,1],
}

X_inputs_val = {
    'LR-LASSO'    : X_val_sc,
    'GAM'         : X_val_sc,
    'XGBoost'     : X_val_sc,
    'RandomForest': X_val_sc,
    'ExtraTrees'  : X_val_sc,
    'MLP'         : X_val_sc,
}

# Metrics summary
results_val = {
    'LR-LASSO'    : lr_val,
    'GAM'         : gam_val,
    'XGBoost'     : xgb_val,
    'RandomForest': rf_val,
    'ExtraTrees'  : et_val,
    'MLP'         : mlp_val,
}

header = f"\n{'Model':<14} {'AUC':>6} {'Sens':>6} {'Spec':>6} {'Prec':>6} {'NPV':>6} {'F1':>6}"
print("\n" + "="*56)
print("METRICS SUMMARY ON VALIDATION SET")
print("="*56)
print(header)
print("─"*56)
for name, m in results_val.items():
    print(f"{name:<14} "
          f"{m['AUC']:>6.2f} {m['Sens']:>6.2f} {m['Spec']:>6.2f} "
          f"{m['Prec']:>6.2f} {m['NPV']:>6.2f} {m['F1']:>6.2f}")
print("─"*56)

# Save validation metrics
rows = []
for name, m in results_val.items():
    rows.append({'model': name, **m})
pd.DataFrame(rows).to_csv(os.path.join(RESULTS_DIR, 'metrics_val.csv'), index=False)

# ROC curves
fig, ax = plt.subplots(figsize=(7, 6))

for name, fn in proba_fn.items():
    X_in = X_inputs_val[name]
    probs = fn(X_in)
    RocCurveDisplay.from_predictions(
        y_val, probs,
        ax=ax,
        name=f"{name})"
    )

ax.plot([0,1],[0,1],'k--', lw=0.8)
ax.set_title("ROC Curves - Validation Set")
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/roc_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# GAM Interpretation

Step 0: Complete Summary

In [ ]:
gam.summary()

Step 1:  Check significant features from GAM summary (p < 0.001, ***)
\
*-> not included in final discussion since [pyGAM](https://pygam.readthedocs.io/en/latest/) p-values are not  reliable*

In [ ]:
# Significant features from GAMs summary (p < 0.001, ***)
significant_idx = [0, 1, 7, 12, 14, 31, 33, 35]  # indexes with *** -> manually checked  indexes -> they obviously change by changing data
significant_features = [kept_spe[i] for i in significant_idx]
print("Significant Features (***) from GAM:")
for f in significant_features:
    print(f" {f}")

Step 2:  Show all partial dependence plots

In [ ]:
fig, axes = plt.subplots(6, 6, figsize=(15, 12))
axes = axes.flatten()

for i, ax in enumerate(axes[:len(kept_spe)]):
    try:
        XX = gam.generate_X_grid(term=i)
        ax.plot(XX[:, i], gam.partial_dependence(term=i, X=XX))
        ax.axhline(0, color='gray', lw=0.5, linestyle='--')
        ax.set_title(kept_spe[i].replace('original_', ''), fontsize=7)
    except ValueError:
        ax.set_visible(False)
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'partial_dependence_gam.png'), dpi=150)
plt.show()

Step 3: Perform Permutation Importance

In [ ]:
# Permutation Importance
baseline_auc = roc_auc_score(y_val, gam.predict_proba(X_val_sc))
print(f"Baseline AUC: {baseline_auc:.4f}")

n_repeats = 50
importances = np.zeros((len(kept_spe), n_repeats))

for j in range(len(kept_spe)):
    for r in range(n_repeats):
        X_permuted = X_val_sc.copy()
        # permute column j
        X_permuted[:, j] = np.random.permutation(X_permuted[:, j])
        auc_permuted = roc_auc_score(y_val, gam.predict_proba(X_permuted))
        importances[j, r] = baseline_auc - auc_permuted

imp_mean = importances.mean(axis=1)
imp_std  = importances.std(axis=1)

imp_df = pd.DataFrame({
    'feature'   : kept_spe,
    'importance': imp_mean,
    'std'       : imp_std
}).sort_values('importance', ascending=False)

print("\nTop 10 features by permutation importance:")
for i, row in imp_df.head(10).iterrows():
    print(f"  {row['feature']:<50} {row['importance']:.4f} ± {row['std']:.4f}")

# Plot
plt.figure(figsize=(10, 6))
top10 = imp_df.head(10).sort_values('importance')
plt.barh(top10['feature'].str.replace('original_', ''),
         top10['importance'],
         xerr=top10['std'],
         color='steelblue', capsize=3)
plt.xlabel('Mean AUC decrease')
plt.title('Permutation Importance — GAM (top 10)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'permutation_importance_gam.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save
imp_df.to_csv(os.path.join(RESULTS_DIR, 'permutation_importance.csv'), index=False)

Step 4: Check Spearman correlation between top 10 features by permutation importance

In [ ]:
#  Top 10 features
top10_features = imp_df.head(10)['feature'].tolist()
top10_idx = [list(kept_spe).index(f) for f in top10_features]
X_top10 = X_val_sc[:, top10_idx]

# Compute Spearman correlation matrix
corr_matrix = np.zeros((10, 10))
for i in range(10):
    for j in range(10):
        corr_matrix[i, j], _ = spearmanr(X_top10[:, i], X_top10[:, j])

# Names for the plot
short_names = [f.replace('original_', '').replace('firstorder_', 'FO_')
                .replace('shape2D_', 'Shape_').replace('glcm_', 'GLCM_')
                .replace('glrlm_', 'GLRLM_') for f in top10_features]

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix,
            xticklabels=short_names,
            yticklabels=short_names,
            annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1,
            center=0)
plt.title('Spearman correlation — top 10 features')
plt.tight_layout()
plt.show()
plt.savefig(os.path.join(RESULTS_DIR, 'spearman_top10.png'), dpi=150, bbox_inches='tight')

Step 5: Partial Dependence Plots of the top 3 features by interpretability (Check [Salmanpour](https://arxiv.org/abs/2603.22692) paper)

In [ ]:
#  Update top3_features based on permutation importance results above
top3_features = ['original_firstorder_Mean', 'original_firstorder_Entropy', 'original_shape2D_Elongation']
top3_idx = [list(kept_spe).index(f) for f in top3_features]

labels = ['Mean\n(First Order)', 'Entropy\n(First Order)', 'Elongation\n(Shape2D)']

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
axes = axes.flatten()

for plot_idx, (feat, idx) in enumerate(zip(top3_features, top3_idx)):
    ax = axes[plot_idx]
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

    XX = gam.generate_X_grid(term=idx)
    pdep, confi = gam.partial_dependence(term=idx, X=XX, width=0.95)

    ax.plot(XX[:, idx], pdep, color='steelblue', linewidth=2)
    ax.fill_between(XX[:, idx], confi[:, 0], confi[:, 1],
                    alpha=0.2, color='steelblue')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1.5)
    ax.set_title(labels[plot_idx], fontweight='bold', fontsize=16)
    ax.set_xlabel('Feature value', fontsize=12)
    ax.set_ylabel('f(x) - log-odds contribution', fontsize=14)
    ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gam_pdp_top3.png'), dpi=150, bbox_inches='tight')
plt.show()

#  Label-Conditional Cross-Conformal Prediction (CCCP)
*Framework: [Vovk 2015](https://link.springer.com/article/10.1007/s10472-013-9368-4), Conformity Score: Log odds*

Step 0:  Data Loading
\
*Run only if you skipped the Classification section above*

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

kept_spe   = np.load(os.path.join(RESULTS_DIR, 'kept_spe.npy'))
X_train_sc = np.load(os.path.join(RESULTS_DIR, 'X_train_sc.npy'))
X_val_sc   = np.load(os.path.join(RESULTS_DIR, 'X_val_sc.npy'))
X_test_sc  = np.load(os.path.join(RESULTS_DIR, 'X_test_sc.npy'))
y_train    = np.load(os.path.join(RESULTS_DIR, 'y_train.npy'))
y_val      = np.load(os.path.join(RESULTS_DIR, 'y_val.npy'))
y_test     = np.load(os.path.join(RESULTS_DIR, 'y_test.npy'))

metrics_df = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics_val.csv'))

print("Loading finished.")

Step 1. Base settings

In [ ]:
# Hyperparameters
K     = 10
alpha = 0.1
SEED  = 42
n_test = len(y_test)

CLASSIFIER = 'gam'   # 'gam' | 'lr' | 'xgb' | 'rf' | 'et'

num_0 = np.zeros(n_test)
num_1 = np.zeros(n_test)

X_trainval = np.vstack([X_train_sc, X_val_sc])
y_trainval = np.concatenate([y_train, y_val])

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

In [ ]:
# Model fit function based on classifier
def get_model(name, y_tr):
    if name == 'gam':
        return LogisticGAM(n_splines=10)
    elif name == 'lr':
        C = lasso_selector.C_[0]
        return LogisticRegression(C=C, penalty='l1', solver='saga',
                                  class_weight='balanced', max_iter=10000)
    elif name == 'xgb':
        scale_pos = (y_tr==0).sum() / (y_tr==1).sum()
        return XGBClassifier(n_estimators=500, learning_rate=0.05,
                             max_depth=4, scale_pos_weight=scale_pos,
                             random_state=42, n_jobs=-1,
                             eval_metric='logloss', verbosity=0)
    elif name == 'rf':
        return RandomForestClassifier(n_estimators=500,
                                      class_weight='balanced',
                                      random_state=42, n_jobs=-1)
    elif name == 'et':
        return ExtraTreesClassifier(n_estimators=500,
                                    class_weight='balanced',
                                    random_state=42, n_jobs=-1)



Step 2: Conformal Prediciton Calibration

In [ ]:
# Conformity Calibration Loop
for fold, (train_idx, cal_idx) in enumerate(
        skf.split(X_trainval, y_trainval)):
    print(f"  Fold {fold+1}/{K}...", end=' ')

    X_tr,  y_tr  = X_trainval[train_idx], y_trainval[train_idx]
    X_cal, y_cal = X_trainval[cal_idx],   y_trainval[cal_idx]

    model_k = get_model(CLASSIFIER, y_tr)
    if CLASSIFIER == 'gam':
      sw = compute_sample_weight('balanced', y_tr)
      model_k.fit(X_tr, y_tr, weights=sw)
    else:
      model_k.fit(X_tr, y_tr)

# Log Odds Conformity Score (Vovk)
    if CLASSIFIER == 'gam':
      prob_cal = model_k.predict_proba(X_cal)
      prob_test_k = model_k.predict_proba(X_test_sc)
    else:
      prob_cal = model_k.predict_proba(X_cal)[:,1]
      prob_test_k = model_k.predict_proba(X_test_sc)[:,1]

    logodds_cal  = np.log(prob_cal / (1 - prob_cal + 1e-8))
    scores_cal_k = np.where(y_cal == 1, logodds_cal, -logodds_cal)

    logodds_test = np.log(prob_test_k / (1 - prob_test_k + 1e-8))
    cscore_test_0 = -logodds_test
    cscore_test_1 =  logodds_test

    ''' If you want to use 1-LAC as conformity score instead of log odds (same output)
    prob_cal_2d  = np.stack([1-prob_cal, prob_cal], axis=1)
    scores_cal_k = prob_cal_2d[np.arange(len(y_cal)), y_cal]
    prob_test_k  = model_k.predict_proba(X_test_spe)[:,1]
    cscore_test_0 = 1 - prob_test_k
    cscore_test_1 = prob_test_k
    '''

    # Vovk formula (9)
    # high conformity = conform -> many cal satisfy it -> high p-value -> include
    scores_k_0 = scores_cal_k[y_cal == 0]
    scores_k_1 = scores_cal_k[y_cal == 1]

    num_0 += (scores_k_0[:, None] <= cscore_test_0[None, :]).sum(axis=0)
    num_1 += (scores_k_1[:, None] <= cscore_test_1[None, :]).sum(axis=0)

    print(f"Average conformity score: {scores_cal_k.mean():.3f}")

# Label-conditional denominators
denom_0 = (y_trainval == 0).sum() + 1
denom_1 = (y_trainval == 1).sum() + 1

pval_0 = (num_0 + 1) / denom_0
pval_1 = (num_1 + 1) / denom_1

print(f"\n p-value means - p^0: {pval_0.mean():.3f}  p^1: {pval_1.mean():.3f}")
print(f"p^0: min={pval_0.min():.4f}  max={pval_0.max():.4f}")
print(f"p^1: min={pval_1.min():.4f}  max={pval_1.max():.4f}")

# Verify: score cal and test now have to be distributed similarly
print(f"\n Conformity Score on Calibration - mean: {scores_cal_k.mean():.3f}  "
      f"Median: {np.median(scores_cal_k):.3f}")


Step 3: Conformity scores distribution

In [ ]:
print(f"\n{'='*55}")
print(f"CONFORMITY SCORES DISTRIBUTION - {CLASSIFIER.upper()}")
print(f"{'='*55}")

print(f"Benign Scores - mean: {scores_k_0.mean():.3f}  "
      f"std: {scores_k_0.std():.3f}")
print(f"Malignant Scores - mean: {scores_k_1.mean():.3f}  "
      f"std: {scores_k_1.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(scores_k_0, bins=30, color='steelblue', alpha=0.7)
axes[0].set_title(f'Benign Score — {CLASSIFIER.upper()} (trainval)')
axes[0].set_xlabel('Conformity score')
axes[1].hist(scores_k_1, bins=30, color='tomato', alpha=0.7)
axes[1].set_title(f'Malignant Score - {CLASSIFIER.upper()} (trainval)')
axes[1].set_xlabel('Conformity score')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'score_dist_cccp_{CLASSIFIER}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# prediction sets
prediction_sets = []
for i in range(n_test):
    pred_set = []
    if pval_0[i] > alpha:
        pred_set.append(0)
    if pval_1[i] > alpha:
        pred_set.append(1)
    prediction_sets.append(pred_set)


Step 4: Conformal metrics

In [ ]:
certain_benign = [(i,ps) for i,ps in enumerate(prediction_sets) if ps==[0]]
certain_malign = [(i,ps) for i,ps in enumerate(prediction_sets) if ps==[1]]
uncertain      = [(i,ps) for i,ps in enumerate(prediction_sets) if len(ps)==2]
empty          = [(i,ps) for i,ps in enumerate(prediction_sets) if len(ps)==0]

coverage_marginal = np.mean([y_test[i] in ps
                             for i,ps in enumerate(prediction_sets)])
idx_ben = np.where(y_test == 0)[0]
idx_mal = np.where(y_test == 1)[0]
coverage_benign = np.mean([0 in prediction_sets[i] for i in idx_ben])
coverage_malign = np.mean([1 in prediction_sets[i] for i in idx_mal])
uncertain_rate  = np.mean([len(ps)==2 for ps in prediction_sets])
efficiency      = np.mean([len(ps)==1 for ps in prediction_sets])

npv_cond  = np.mean([y_test[i]==0 for i,_ in certain_benign]) \
            if certain_benign else None
prec_cond = np.mean([y_test[i]==1 for i,_ in certain_malign]) \
            if certain_malign else None

print(f"\n{'='*55}")
print(f"CCCP - {CLASSIFIER.upper()}, K={K}, α={alpha}")
print(f"{'='*55}")
print(f"Total Coverage: {coverage_marginal:.3f}  (target ≥ {1-alpha:.2f})")
print(f"Benign Coverage: {coverage_benign:.3f}  (target ≥ {1-alpha:.2f})")
print(f"Malignant Coverage    : {coverage_malign:.3f}  (target ≥ {1-alpha:.2f})")
print(f"Efficiency         : {efficiency:.3f}")
print(f"Uncertain rate     : {uncertain_rate:.3f}")
print(f"\n{{Benign}}          : {len(certain_benign):4d} ({len(certain_benign)/n_test*100:.1f}%)")
print(f"{{Malignant}}          : {len(certain_malign):4d} ({len(certain_malign)/n_test*100:.1f}%)")
print(f"{{Benign, Malignant}} : {len(uncertain):4d} ({len(uncertain)/n_test*100:.1f}%)")
print(f"{{}} empty           : {len(empty):4d} ({len(empty)/n_test*100:.1f}%)")
if npv_cond is not None:
    print(f"\nConditional NPV    : {npv_cond:.4f} (on {len(certain_benign)} certain benign)")
if prec_cond is not None:
    print(f"Conditional Precision  : {prec_cond:.4f} (on {len(certain_malign)} certain malignant)")


Step 5: Prediction examples

In [ ]:
print(f"\n{'='*55}")
print("Prediction examples (first 20 test samples)")
print(f"{'='*55}")
print(f"{'ID':>4} {'True Label':>12} {'Pred set':>20} "
      f"{'P(malignant)':>12} {'Outcome':>10}")
print("-"*55)

label_names = {0: 'Benign', 1: 'Malignant'}
for i in range(100):
    true = y_test[i]
    ps   = prediction_sets[i]
    prob_mal = prob_test_k[i]

    if len(ps) == 0:
        ps_str = '{}'
        output  = 'ANOMALY'
    elif len(ps) == 2:
        ps_str = '{B, M}'
        output  = 'UNCERTAIN'
    elif ps == [0]:
        ps_str = '{Benign}'
        output  = '✓' if true == 0 else 'FN!'
    else:
        ps_str = '{Malignant}'
        output  = '✓' if true == 1 else 'FP'

    print(f"{i:>4} {label_names[true]:>12} {ps_str:>20} "
          f"{prob_mal:>12.3f} {output:>10}")
